<a href="https://colab.research.google.com/github/thunderish48-pixel/demo-verify-engine/blob/main/verify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:

# =========================================================
# VERIFY PIPELINE (FINAL - STABLE, NO TIMEOUT, SINGLE CELL)
# =========================================================

# HARD RESET
try:
    client.close()
except:
    pass

!pip install -q hedera-sdk-py

import json, time, hashlib
from hedera import (
    Client,
    TopicCreateTransaction,
    TopicMessageSubmitTransaction,
    TopicMessageQuery,
    AccountId,
    PrivateKey,
    TopicId
)
from jnius import autoclass # Import autoclass

# -------------------------------
# CONFIG
# -------------------------------
ACCOUNT_ID = AccountId.fromString("0.0.8253770")
PRIVATE_KEY = PrivateKey.fromString("3030020100300706052b8104000a04220420d9eb6d8c1cd6efaefaefb44e6052fe82c834cd87eefa8c4c5b2b61ace32d8fbb")
TOPIC_ID = "0.0.8261620"  # your topic

# -------------------------------
# INIT
# -------------------------------
client = Client.forTestnet()
client.setOperator(ACCOUNT_ID, PRIVATE_KEY)

# -------------------------------
# TOPIC
# -------------------------------
if TOPIC_ID is None:
    tx = TopicCreateTransaction().execute(client)
    receipt = tx.getReceipt(client)
    topic_id = receipt.topicId
    print(f"[INFO] Created Topic: {topic_id}")
else:
    topic_id = TopicId.fromString(TOPIC_ID)
    print(f"[INFO] Using Existing Topic: {topic_id}")

# -------------------------------
# EVENT + HASH
# -------------------------------
event = {
    "event_id": "001",
    "system": "verification_pipeline",
    "timestamp": int(time.time()),
    "status": "validated"
}

event_str = json.dumps(event)
event_hash = hashlib.sha256(event_str.encode()).hexdigest()

print(f"[INFO] Event Hash: {event_hash}")

# -------------------------------
# VERIFY STATE
# -------------------------------
verified = {"status": False}
subscription = {"ref": None}

# -------------------------------
# HANDLER (ROBUST)
# -------------------------------
def handle(msg):
    try:
        message = bytes(msg.contents)
        decoded = message.decode("utf-8", errors="ignore")

        if event_hash in decoded:
            print(f"[VERIFIED] {decoded}")
            verified["status"] = True

            if subscription["ref"] is not None:
                subscription["ref"].unsubscribe()

    except Exception as e:
        print("[HANDLER ERROR]", e)

# -------------------------------
# SUBSCRIBE FIRST (IMPORTANT)
# -------------------------------
# Get the Java Instant class directly via jnius
Instant = autoclass('java.time.Instant')

subscription["ref"] = TopicMessageQuery() \
    .setTopicId(topic_id) \
    .setStartTime(Instant.ofEpochSecond(int(time.time()) - 5)) \
    .subscribe(client, handle)

time.sleep(1)  # ensure listener is active

# -------------------------------
# SUBMIT
# -------------------------------
TopicMessageSubmitTransaction() \
    .setTopicId(topic_id) \
    .setMessage(event_hash.encode()) \
    .execute(client) \
    .getReceipt(client)

print("[INFO] Submitted")
print("[INFO] Verifying...")

# -------------------------------
# WAIT LOOP
# -------------------------------
timeout = 20
start = time.time()

while time.time() - start < timeout:
    if verified["status"]:
        break
    time.sleep(0.5)

if not verified["status"]:
    print("[WARNING] Verification timeout")
else:
    print(f"[VERIFIED] Hash confirmed on Hedera: {event_hash}")

[INFO] Using Existing Topic: <com.hedera.hashgraph.sdk.TopicId at 0x784c0eb2e580 jclass=com/hedera/hashgraph/sdk/TopicId jself=<LocalRef obj=0x261384f0 at 0x784b6ec73570>>
[INFO] Event Hash: bfb4d24f4fbab5c4dc6ecbb92ef72b39babcce33a032416d4af7c40b93a25a9d
[INFO] Submitted
[INFO] Verifying...
[VERIFIED] bfb4d24f4fbab5c4dc6ecbb92ef72b39babcce33a032416d4af7c40b93a25a9d
[VERIFIED] Hash confirmed on Hedera: bfb4d24f4fbab5c4dc6ecbb92ef72b39babcce33a032416d4af7c40b93a25a9d
